## **Breast Cancer Classification with Activication Function Comparison**

Implement a basic neural network from scratch using NumPy to classify
binary data, demonstrate and compare activation functions like Sigmoid, ReLU, and Tanh,
manually implement gradient descent and backpropagation with L1 and L2 regularization,
and experiment with hyperparameter tuning such as learning rate, epochs, and batch size
while documenting the results

In [2]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

In [9]:
df = pd.read_csv("/content/Breast_cancer_dataset.csv")

In [10]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [11]:
df.tail()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
564,926424,M,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,...,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,NaN
565,926682,M,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,...,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,NaN
566,926954,M,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,...,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,NaN
567,927241,M,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,...,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,NaN
568,92751,B,7.76,24.54,47.92,181.0,0.05263,0.04362,0.00000,0.00000,...,30.37,59.16,268.6,0.08996,0.06444,0.0000,0.0000,0.2871,0.07039,NaN


In [12]:
df.shape

(569, 33)

In [13]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,569.0,3.037183e+07,1.250206e+08,8670.000000,869218.000000,906024.000000,8.813129e+06,9.113205e+08
radius_mean,569.0,1.412729e+01,3.524049e+00,6.981000,11.700000,13.370000,1.578000e+01,2.811000e+01
texture_mean,569.0,1.928965e+01,4.301036e+00,9.710000,16.170000,18.840000,2.180000e+01,3.928000e+01
perimeter_mean,569.0,9.196903e+01,2.429898e+01,43.790000,75.170000,86.240000,1.041000e+02,1.885000e+02
area_mean,569.0,6.548891e+02,3.519141e+02,143.500000,420.300000,551.100000,7.827000e+02,2.501000e+03
smoothness_mean,569.0,9.636028e-02,1.406413e-02,0.052630,0.086370,0.095870,1.053000e-01,1.634000e-01
compactness_mean,569.0,1.043410e-01,5.281276e-02,0.019380,0.064920,0.092630,1.304000e-01,3.454000e-01
concavity_mean,569.0,8.879932e-02,7.971981e-02,0.000000,0.029560,0.061540,1.307000e-01,4.268000e-01
concave points_mean,569.0,4.891915e-02,3.880284e-02,0.000000,0.020310,0.033500,7.400000e-02,2.012000e-01
symmetry_mean,569.0,1.811619e-01,2.741428e-02,0.106000,0.161900,0.179200,1.957000e-01,3.040000e-01


In [14]:
# Convert diagnosis
# M=1, B=0

df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    int64  
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [16]:
# Drop unnecessary columns
df.drop(["id", "Unnamed: 32"], axis=1, inplace=True)

In [17]:
df

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,1,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,1,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,1,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,1,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [18]:
# Features & target
X = df.drop("diagnosis", axis=1).values
y = df["diagnosis"].values.reshape(-1, 1)

### **Train-Test Split & Scaling**

In [19]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

### **Activation Functions**

In [20]:
def sigmoid(z):
    return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1 - s)

def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def tanh(z):
    return np.tanh(z)

def tanh_derivative(z):
    return 1 - np.tanh(z) ** 2

### **Initialize Parameters**

In [21]:
def init_params(n_x, n_h, n_y):
    W1 = np.random.randn(n_x, n_h) * 0.01
    b1 = np.zeros((1, n_h))
    W2 = np.random.randn(n_h, n_y) * 0.01
    b2 = np.zeros((1, n_y))
    return W1, b1, W2, b2

### **Forward Propagation**

In [22]:
def forward(X, W1, b1, W2, b2, act):
    Z1 = np.dot(X, W1) + b1

    if act == "relu":
        A1 = relu(Z1)
    elif act == "tanh":
        A1 = tanh(Z1)
    else:
        A1 = sigmoid(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = sigmoid(Z2)

    return Z1, A1, Z2, A2

### **Loss Function (L1 + L2)**

In [23]:
def loss_fn(y, A2, W1, W2, reg=None, lam=0.01):
    m = y.shape[0]

    loss = -(1/m) * np.sum(y*np.log(A2+1e-8) + (1-y)*np.log(1-A2+1e-8))

    if reg == "L2":
        loss += (lam/(2*m)) * (np.sum(W1**2) + np.sum(W2**2))

    if reg == "L1":
        loss += (lam/(2*m)) * (np.sum(np.abs(W1)) + np.sum(np.abs(W2)))

    return loss

### **Backpropagation**

In [24]:
def backward(X, y, Z1, A1, A2, W2, act, reg=None, lam=0.01):
    m = X.shape[0]

    dZ2 = A2 - y
    dW2 = (1/m) * np.dot(A1.T, dZ2)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, W2.T)

    if act == "relu":
        dZ1 = dA1 * relu_derivative(Z1)
    elif act == "tanh":
        dZ1 = dA1 * tanh_derivative(Z1)
    else:
        dZ1 = dA1 * sigmoid_derivative(Z1)

    dW1 = (1/m) * np.dot(X.T, dZ1)
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

    # Regularization
    if reg == "L2":
        dW2 += (lam/m) * W2
        dW1 += (lam/m) * W1

    if reg == "L1":
        dW2 += (lam/m) * np.sign(W2)
        dW1 += (lam/m) * np.sign(W1)

    return dW1, db1, dW2, db2

### **Training**

In [25]:
def create_batches(X, y, batch_size):
    m = X.shape[0]
    indices = np.random.permutation(m)

    for i in range(0, m, batch_size):
        batch_idx = indices[i:i+batch_size]
        yield X[batch_idx], y[batch_idx]

In [26]:
def train(X, y, act="relu", lr=0.01, epochs=1000, batch_size=32, reg=None):
    n_x = X.shape[1]
    n_h = 16
    n_y = 1

    W1, b1, W2, b2 = init_params(n_x, n_h, n_y)

    losses = []

    for epoch in range(epochs):
        for X_batch, y_batch in create_batches(X, y, batch_size):

            Z1, A1, Z2, A2 = forward(X_batch, W1, b1, W2, b2, act)

            loss = loss_fn(y_batch, A2, W1, W2, reg)
            losses.append(loss)

            dW1, db1, dW2, db2 = backward(
                X_batch, y_batch, Z1, A1, A2, W2, act, reg
            )

            # Update
            W1 -= lr * dW1
            b1 -= lr * db1
            W2 -= lr * dW2
            b2 -= lr * db2

        if epoch % 100 == 0:
            print(f"{act} | Epoch {epoch} | Loss: {loss}")

    return W1, b1, W2, b2, losses

### **Prediction & Accuracy**

In [27]:
def predict(X, W1, b1, W2, b2, act):
    _, _, _, A2 = forward(X, W1, b1, W2, b2, act)
    return (A2 > 0.5).astype(int)

### **Activation Function Comparison**

In [28]:
# Activation comparison
for act in ["sigmoid", "relu", "tanh"]:
    print(f"\n=== Activation: {act} ===")

    W1, b1, W2, b2, _ = train(X_train, y_train, act=act)

    preds = predict(X_test, W1, b1, W2, b2, act)
    print("Accuracy:", accuracy_score(y_test, preds))


=== Activation: sigmoid ===
sigmoid | Epoch 0 | Loss: 0.6883630661337459
sigmoid | Epoch 100 | Loss: 0.3051678887458838
sigmoid | Epoch 200 | Loss: 0.14155049578460258
sigmoid | Epoch 300 | Loss: 0.10836651309628126
sigmoid | Epoch 400 | Loss: 0.037426587596448534
sigmoid | Epoch 500 | Loss: 0.03592885411907278
sigmoid | Epoch 600 | Loss: 0.005693139504407992
sigmoid | Epoch 700 | Loss: 0.0038390073980309764
sigmoid | Epoch 800 | Loss: 0.04390227580113853
sigmoid | Epoch 900 | Loss: 0.003857832665831695
Accuracy: 0.9824561403508771

=== Activation: relu ===
relu | Epoch 0 | Loss: 0.6894697728429389
relu | Epoch 100 | Loss: 0.10204752340456244
relu | Epoch 200 | Loss: 0.1004530567027476
relu | Epoch 300 | Loss: 0.01766382982147211
relu | Epoch 400 | Loss: 0.01298797973392541
relu | Epoch 500 | Loss: 0.16339593815085085
relu | Epoch 600 | Loss: 0.0008449405970501324
relu | Epoch 700 | Loss: 0.004715456976444305
relu | Epoch 800 | Loss: 0.007581807313883934
relu | Epoch 900 | Loss: 0.006

In [29]:
# Regularization comparison
for reg in [None, "L1", "L2"]:
    print(f"\n=== Regularization: {reg} ===")

    W1, b1, W2, b2, _ = train(X_train, y_train, reg=reg)

    preds = predict(X_test, W1, b1, W2, b2, "relu")
    print("Accuracy:", accuracy_score(y_test, preds))


=== Regularization: None ===
relu | Epoch 0 | Loss: 0.6940420255472609
relu | Epoch 100 | Loss: 0.03800697212579036
relu | Epoch 200 | Loss: 0.005904622087606865
relu | Epoch 300 | Loss: 0.023267390657140185
relu | Epoch 400 | Loss: 0.008016854653006706
relu | Epoch 500 | Loss: 0.016232146161031743
relu | Epoch 600 | Loss: 0.02178033688975581
relu | Epoch 700 | Loss: 0.003002865330099821
relu | Epoch 800 | Loss: 0.004664114225290645
relu | Epoch 900 | Loss: 0.010858707551937356
Accuracy: 0.9824561403508771

=== Regularization: L1 ===
relu | Epoch 0 | Loss: 0.6945923080064486
relu | Epoch 100 | Loss: 0.05586910574342102
relu | Epoch 200 | Loss: 0.06068719495753458
relu | Epoch 300 | Loss: 0.22991447447645053
relu | Epoch 400 | Loss: 0.08776336954097075
relu | Epoch 500 | Loss: 0.03594394353149072
relu | Epoch 600 | Loss: 0.04064590356289083
relu | Epoch 700 | Loss: 0.04190505974805596
relu | Epoch 800 | Loss: 0.042629152976528376
relu | Epoch 900 | Loss: 0.06610253549749787
Accuracy: 0

In [30]:
# Learning rate tuning
for lr in [0.1, 0.01, 0.001]:
    print(f"\nLearning Rate: {lr}")

    W1, b1, W2, b2, _ = train(X_train, y_train, lr=lr)

    preds = predict(X_test, W1, b1, W2, b2, "relu")
    print("Accuracy:", accuracy_score(y_test, preds))


Learning Rate: 0.1
relu | Epoch 0 | Loss: 0.6613428215227668
relu | Epoch 100 | Loss: 0.011940011521024179
relu | Epoch 200 | Loss: 0.001408232939787397
relu | Epoch 300 | Loss: 0.00017667036354672848
relu | Epoch 400 | Loss: 0.0021559082914784397
relu | Epoch 500 | Loss: 0.00016948025592821364
relu | Epoch 600 | Loss: 0.0026570238514293694
relu | Epoch 700 | Loss: 0.0019118871426538986
relu | Epoch 800 | Loss: 0.0009465018962485243
relu | Epoch 900 | Loss: -9.027535818616936e-09
Accuracy: 0.9649122807017544

Learning Rate: 0.01
relu | Epoch 0 | Loss: 0.6940075277321287
relu | Epoch 100 | Loss: 0.0802915132318825
relu | Epoch 200 | Loss: 0.033617050776303385
relu | Epoch 300 | Loss: 0.0022017565781246324
relu | Epoch 400 | Loss: 0.07128642823867225
relu | Epoch 500 | Loss: 0.009693812290582733
relu | Epoch 600 | Loss: 0.0023580087539399395
relu | Epoch 700 | Loss: 0.029386478104929885
relu | Epoch 800 | Loss: 0.02696957327976495
relu | Epoch 900 | Loss: 0.021005500879344153
Accuracy: 

## **Plot loss**

In [31]:
# PLOT LOSS

W1, b1, W2, b2, losses = train(X_train, y_train)

plt.plot(losses)
plt.title("Loss Curve")
plt.xlabel("Iterations")
plt.ylabel("Loss")
plt.show()

relu | Epoch 0 | Loss: 0.6914818228190672
relu | Epoch 100 | Loss: 0.044925834514420254
relu | Epoch 200 | Loss: 0.02341553603910545
relu | Epoch 300 | Loss: 0.032749465436779514
relu | Epoch 400 | Loss: 0.027863653877649205
relu | Epoch 500 | Loss: 0.3253461023727874
relu | Epoch 600 | Loss: 0.014387074800964302
relu | Epoch 700 | Loss: 0.0022517685419702164
relu | Epoch 800 | Loss: 0.0007788222032258998
relu | Epoch 900 | Loss: 0.003773813170974001
